pour chaque mois et chaque cellule tarifaire, on tire un nombre de sinistres et un coût à partir du modèle, puis on introduit une dérive à partir du mois 7 (fréquence des 18-25 ans × 1,4). Ce 1,4 est un choix arbitraire, uniquement pour avoir un exemple à détecter.

In [3]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/portfolio_priced.parquet")

cells = (df.groupby(["age_group", "bm_group"])
           .agg(exposition=("Exposure", "sum"),
                freq_pred=("freq_pred", "first"),
                sev_pred=("sev_pred", "first"))
           .reset_index())

rng = np.random.default_rng(42)
k = 0.5          # forme de la loi Gamma des coûts (plus k est petit, plus c'est dispersé)
rows = []

for mois in range(1, 13):
    expo_m = cells["exposition"] / 12
    drift = np.where((cells["age_group"] == "(17, 25]") & (mois >= 7), 1.4, 1.0)

    attendus = cells["freq_pred"] * expo_m                  # sinistres attendus par le modèle
    N = rng.poisson(attendus * drift)                       # sinistres réellement simulés
    cout = np.where(N > 0, rng.gamma(np.maximum(N, 1) * k, cells["sev_pred"] / k), 0.0)

    m = cells[["age_group", "bm_group"]].copy()
    m["mois"] = mois
    m["exposition"] = expo_m
    m["sin_attendus"] = attendus
    m["sin_reels"] = N
    m["prime"] = attendus * cells["sev_pred"]               # prime pure sans le chargement grands sinistres
    m["cout"] = cout
    rows.append(m)

sim = pd.concat(rows, ignore_index=True)
sim.to_csv("../data/monitoring_sim.csv", index=False)
sim.shape


(432, 8)

Ce que fait cette simulation, et ses limites
Les coûts suivent une loi Gamma, sans les gros sinistres (on a mis ceux-ci dans charge_grands). La prime utilisée est donc fréquence × sévérité, sans ce chargement, et le S/P doit tourner autour de 100 % tant qu’il n’y a pas de dérive.
Le nombre de sinistres est un Poisson pur, sans la surdispersion de 1,75 mesurée plus haut. Les fluctuations simulées seront donc un peu plus faibles qu’en réalité.
Les polices sont fixes. Rien ne simule les entrées et les sorties du portefeuille.

In [4]:
def sp(g):
    return g["cout"].sum() / g["prime"].sum()

def ae(g):
    return g["sin_reels"].sum() / g["sin_attendus"].sum()

jeunes = sim[sim["age_group"] == "(17, 25]"]
autres = sim[sim["age_group"] != "(17, 25]"]

resume = pd.DataFrame({
    "SP_jeunes": jeunes.groupby("mois").apply(sp),
    "SP_autres": autres.groupby("mois").apply(sp),
    "AE_jeunes": jeunes.groupby("mois").apply(ae),
    "AE_autres": autres.groupby("mois").apply(ae),
})
resume.round(2)

,SP_jeunes,SP_autres,AE_jeunes,AE_autres
mois,,,,
1,0.91,1.02,0.92,0.99
2,0.99,0.98,1.06,1.00
3,0.96,1.03,1.08,1.03
4,0.89,0.99,0.96,1.02
5,1.08,1.00,1.04,0.99
6,1.05,1.04,1.01,1.01
7,1.27,0.95,1.33,0.99
8,1.11,1.04,1.28,1.02
9,1.24,0.95,1.44,0.98
